In [1]:
from science_jubilee.JubileeManager import JubileeManager
from science_jubilee.decks.LoadAll import load_all
from science_jubilee.tools.Loop import *
from science_jubilee.labware.Labware import *
from science_jubilee.decks.Deck import *

In [2]:
# Connexion à la Jubilee
jm = JubileeManager(address="10.0.9.6", simulated=False)
ino = Loop


2026-03-06 11:54:45 - [JubileeController]  - INFO - Initializing JubileeController (simulated=False, address=10.0.9.6)
2026-03-06 11:54:45 - [JubileeController]  - WARNING - Disconnecting this application from the network will halt connection to Jubilee.
2026-03-06 11:54:45 - [JubileeController]  - INFO - Connecting to Jubilee machine...
2026-03-06 11:54:45 - [JubileeController]  - INFO - Successfully connected and initialized Jubilee machine.
2026-03-06 11:54:45 - [JubileeManager]     - INFO - JubileeManager initialized (simulated=False, max_tools=4).


In [3]:
# Homing
jm.controller.home_all()

Is a tool currently mounted? [y/n]  n
Is the deck clear of any obstacles? [y/n]  y


2026-03-06 11:55:32 - [JubileeController]  - INFO - All axes homed successfully.


In [4]:
# Descendre le plateau pour positionner le deck
jm.controller.move_to(z=300)
jm.controller.move_to(x=150, y=150)

In [5]:
# Enregistrer le deck
deck = jm.load_deck("test1")
load_all(jm,"test1")

2026-03-06 11:55:45 - [Deck]               - INFO - Loading deck configuration from: D:\Polytech\2025-2026\projet_indus\science-jubilee\src\science_jubilee\decks\deck_definition\test1.json
2026-03-06 11:55:45 - [Deck]               - INFO - Deck 'Experience1' loaded with 1 slots.
2026-03-06 11:55:45 - [JubileeManager]     - INFO - Deck 'Experience1' loaded from 'D:\Polytech\2025-2026\projet_indus\science-jubilee\src\science_jubilee\decks\deck_definition'.
2026-03-06 11:55:45 - [Tool]               - INFO - Tool 'Inoculator' (index 0) initialized.
2026-03-06 11:55:45 - [JubileeManager]     - INFO - Tool 'Inoculator' loaded at index 0.
2026-03-06 11:55:45 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, -43.5, 0).


In [6]:
#affiche les slots
jm.deck.list_slots()
jm.deck.get_all_slot_machine_coordinates()

{'0': (104.12, 89.59)}

In [7]:
jm.status()

{'deck': 'Experience1',
 'tools': [{'index': 0, 'name': 'Inoculator'}],
 'active_tool_index': None,
 'active_tool_name': None,
 'tool_offsets': {0: (0, -43.5, 0)}}

In [8]:
# jm.controller.move_to(x=150, y=150)
jm.controller.move_to(z=50)



In [9]:
# 1. On récupère les infos de ton outil par son nom
dict_outil = jm.get_tool_by_name("Inoculator")
idx = dict_outil['index']

# 2. On prépare l'outil physiquement
jm.controller.pickup_tool_sequence(idx)
jm.set_active_tool(idx)
jm.set_tool_offset(idx, (0, -31.5, 35))

tool2 = jm.tools_list[idx]
tool2.__class__ = Loop
tool2._machine = jm.controller  # On le lie au contrôleur pour les mouvements
tool2.is_active_tool = True     # On valide le décorateur de sécurité

# 4. CORRECTIF : On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 80mm de hauteur pour ne rien toucher
    tool2.safe_z_movement = lambda: jm.controller.move_to(z=80)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{tool2.name}' prêt (Classe: {type(tool2).__name__})")

2026-03-06 11:56:00 - [JubileeController]  - INFO - Starting pickup_tool sequence for tool 0.
2026-03-06 11:56:08 - [JubileeController]  - INFO - pickup_tool_sequence completed for tool 0.
2026-03-06 11:56:08 - [JubileeManager]     - INFO - Tool at index 0 set as active tool.
2026-03-06 11:56:08 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, -31.5, 35).


✅ Correctif safe_z_movement appliqué.
✅ Outil 'Inoculator' prêt (Classe: Loop)


In [10]:
# On récupère les objets Well directement depuis le Deck
# Assure-toi que le slot '0' contient bien ton labware
slot_0 = jm.deck.get_slot('0')

well_B1 = slot_0.labware.wells['B1'] # C'est un OBJET, pas un tuple (x,y,z)
well_A1 = slot_0.labware.wells['A1']

print(f"Puits source : {well_B1} à {well_B1.x, well_B1.y}")
print(f"Puits source : {well_A1} à {well_A1.x, well_A1.y}")




Puits source : Well B1 at coordinates (34.5, 15.0, 2.5) à (34.5, 15.0)
Puits source : Well A1 at coordinates (15.0, 15.0, 2.5) à (15.0, 15.0)


In [12]:
# On s'assure d'être en hauteur avant de commencer
jm.controller.move_to(z=80)

print("🚀 Lancement du transfert de la lentille...")

tool2.transfer(
    source=well_B1,          # L'objet Well source
    destination=well_A1,     # L'objet Well destination
    s=2000,                  # Vitesse de transit (mm/min)
    sweep_x=3,               # Largeur du balayage pour attraper (mm)
    sweep_y=3,               # Longueur du balayage (mm)
    sweep_z=10,              # Hauteur de remontée après capture (mm)
    sweep_speed=150,         # Vitesse lente pour ne pas blesser la lentille
    up_speed=800,            # Vitesse de sortie du puits
    randomize_pickup=False   # On vise précisément le centre
)

print("✅ Transfert terminé !")
jm.controller.move_to(z=300)

🚀 Lancement du transfert de la lentille...
✅ Transfert terminé !
